In [ ]:
import duckdb

conn = duckdb.connect('../data/churn_dev.duckdb')

# Distribution across value tiers
print("=== VALUE TIER DISTRIBUTION ===")
print(conn.execute("""
    SELECT 
        value_tier,
        COUNT(*) as customers,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) as pct,
        ROUND(AVG(cashback_amount), 2) as avg_cashback,
        ROUND(MIN(cashback_amount), 2) as min_cashback,
        ROUND(MAX(cashback_amount), 2) as max_cashback
    FROM main.int_customer_value
    GROUP BY value_tier
    ORDER BY min_cashback DESC
""").df().to_string())

# Cashback percentiles
print("\n=== CASHBACK PERCENTILES ===")
print(conn.execute("""
    SELECT 
        ROUND(PERCENTILE_CONT(0.10) WITHIN GROUP (ORDER BY cashback_amount), 2) as p10,
        ROUND(PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY cashback_amount), 2) as p25,
        ROUND(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY cashback_amount), 2) as p50,
        ROUND(PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY cashback_amount), 2) as p75,
        ROUND(PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY cashback_amount), 2) as p90,
        ROUND(PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY cashback_amount), 2) as p95
    FROM main.int_customer_value
""").df().to_string())

# Churn rate by value tier
print("\n=== CHURN RATE BY VALUE TIER ===")
print(conn.execute("""
    SELECT 
        v.value_tier,
        COUNT(*) as customers,
        ROUND(AVG(s.churn) * 100, 1) as churn_rate_pct
    FROM main.int_customer_value v
    JOIN main.stg_customers s USING (customer_id)
    GROUP BY v.value_tier
    ORDER BY churn_rate_pct DESC
""").df().to_string())

conn.close()


ModuleNotFoundError: No module named 'duckdb'